# PCA and spatial ICA: where does coupling information go?

This notebook keeps the **coupling-only result** and makes its mechanism the main story. It contains: (1) the coupling results from your uploaded run; (2) an exact two-source covariance illustration; (3) generating and recovered spatial maps/time courses; (4) group variance and correlation changes; and (5) held-out decoding over a signal-strength sweep.

**Question:** can the same coupling change appear in PCA component variances and spatial ICA connectivity? This is a controlled illustration, not a reproduction of Onila's data or proof that one method is universally better.

The new sweep uses a separate, independent reference cohort to learn maps, then five-fold supervised CV in an evaluation cohort. No evaluation participant contributes to either map fit. Both methods use the same participants, splits, rank, and classifier. This deliberately replaces the expensive shared-versus-fold-fitted comparison; the original result is preserved separately and labeled.


In [ ]:
# Run once if needed, then restart the kernel:
# %pip install numpy scipy pandas matplotlib scikit-learn nilearn nibabel threadpoolctl
import json, platform, warnings, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import nibabel as nib
import nilearn, sklearn
from nilearn.decomposition import CanICA
from sklearn.decomposition import fastica
from sklearn.exceptions import ConvergenceWarning
from sklearn.utils import check_random_state
from sklearn.decomposition import PCA
from sklearn.model_selection import LeaveOneOut, StratifiedKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from scipy.optimize import linear_sum_assignment
from threadpoolctl import threadpool_limits
from IPython.display import display

SEED = 42
N_SUBJECTS = 26
N_RUNS, RUN_LENGTH = 3, 30
GRID = (8, 8, 3)
K_TRUE = K_FIT = 5
NOISE_SD = .35
LANGUAGES = ['L2']
FEATURE_FAMILIES = ['static', 'connectivity', 'combined']
SCENARIOS = ['null', 'amplitude', 'coupling', 'mixed']
TOP_FEATURES, LOGISTIC_C = 10, 1.0
N_NULL_REPS = 100
N_BENCH_REPS = 5
N_SPLITS = 5
ICA_BACKEND = 'audited' # 'audited' or 'stock'; do not choose based on decoding
ICA_MAX_ITER = 2000
ICA_TOL = 1e-4
CANICA_N_INIT = 5
CANICA_THRESHOLD = None
CANICA_STANDARDIZE = False
AFFINE = np.diag([3.,3.,3.,1.])
MASK = nib.Nifti1Image(np.ones(GRID,dtype=np.uint8),AFFINE)
METHODS = ['PCA', 'AuditedCanICA' if ICA_BACKEND == 'audited' else 'StockCanICA']
SCOPES = ['all_participants', 'training_only']
pd.set_option('display.max_columns', 20)
STRENGTHS = [0., .25, .5, .75, 1.]
N_SWEEP_REPS = 5  # demonstration; increase before drawing comparative conclusions
print(dict(python=platform.python_version(), numpy=np.__version__,
           sklearn=sklearn.__version__, nilearn=nilearn.__version__,
           sweep_cohorts=N_SWEEP_REPS, strengths=STRENGTHS))


## 1. Keep the original coupling result
These are the **reported means in `PCA_ICA_revised_diagnostics.html`**, not newly computed estimates. They are from five common successful coupling cohorts, L2, with maps fitted inside the training folds. “Static” there included means, SD, log-SD, and lag-1 correlation, with up to ten selected features. This table is a historical reference; it is not pooled with the new sweep.


In [ ]:
original_coupling = pd.DataFrame({
    'Feature family': ['Static summaries', 'Connectivity', 'Combined'],
    'PCA': [.957576, .575758, .957576],
    'Audited spatial ICA': [.533333, .963636, .969697],
})
display(original_coupling)
fig, ax = plt.subplots(figsize=(7, 3.5))
original_coupling.set_index('Feature family').plot.bar(ax=ax, rot=0, color=['#3274A1','#D57939'])
ax.axhline(.5,color='gray',ls=':'); ax.set(ylim=(0,1.05),ylabel='Held-out AUC',
    title='Preserved result: coupling-only simulation')
ax.legend(frameon=False); plt.tight_layout(); plt.show()


## 2. The mechanism, without estimation noise
For two unit-variance sources with correlation $\rho$, define $u=(s_1+s_2)/\sqrt{2}$ and $v=(s_1-s_2)/\sqrt{2}$. Then
$$\mathrm{Var}(u)=1+\rho,\quad\mathrm{Var}(v)=1-\rho,\quad\mathrm{Cov}(u,v)=0.$$
A change in source correlation becomes a change in variance in these rotated coordinates. The same second-order information has changed location in the matrix. This is the ideal PCA rotation for the equal-variance two-source case; estimated PCA in the noisy five-source simulation need not match it exactly. Spatial ICA seeks independence of **spatial maps**, so its time courses may remain correlated.


In [ ]:
rho0, rho1 = .15, .70
R = np.array([[1,1],[1,-1]])/np.sqrt(2)
C0 = np.array([[1,rho0],[rho0,1]])
C1 = np.array([[1,rho1],[rho1,1]])
fig, axes = plt.subplots(2,3,figsize=(10,5.8))
for row, (label, rot) in enumerate([('Source coordinates',np.eye(2)),('Sum / difference coordinates',R)]):
    mats=[rot@C0@rot.T,rot@C1@rot.T,rot@(C1-C0)@rot.T]
    for col,(title,M) in enumerate(zip(['Group 0 covariance','Group 1 covariance','Group difference'],mats)):
        ax=axes[row,col];ax.imshow(M,cmap='RdBu_r',vmin=-1.7,vmax=1.7)
        for i in range(2):
            for j in range(2): ax.text(j,i,f'{M[i,j]:.2f}',ha='center',va='center')
        ax.set_title(title);ax.set_xticks([0,1]);ax.set_yticks([0,1])
        if col==0:ax.set_ylabel(label)
fig.suptitle('Coupling can become component variance after rotation')
plt.tight_layout();plt.show()
assert np.allclose(R@C1@R.T,np.diag([1+rho1,1-rho1]))


## 3. Generating model and spatial decomposition
The original five disjoint toy spatial sources, three runs, temporal autocorrelation, noise, and participant amplitude variability are retained. These are toy grid maps, not anatomical brain maps or an exact independent-pixel ICA model. A reference cohort and an evaluation cohort share the generating maps but have independently sampled participants and noise.

For group 1 in L2, strength $a$ changes amplitude by a factor $1+0.65a$ (amplitude/mixed), correlation from $0.15$ to $0.15+0.55a$ (coupling/mixed), or both. Strength zero is the null. Seeds are paired across strengths; points on a curve are therefore dependent.

We retain the previous **AuditedCanICA** extension: spatial CanICA with a larger iteration budget, converged-restart selection, and a logged deflation fallback. It is not stock CanICA. Failed fits remain failures; no seed is replaced. All maps are normalized to unit spatial norm before projection. See the [CanICA group model](https://arxiv.org/abs/1006.2300) for the underlying method.


In [ ]:
def true_maps(seed):
    rng = np.random.default_rng(seed)
    coords = np.indices(GRID).reshape(3, -1).T
    centers = np.array([[1,1,1], [1,6,1], [4,3,1], [6,1,1], [6,6,1]])
    d2 = ((coords[:,None,:] - centers[None,:,:])**2).sum(axis=2)
    region = d2.argmin(axis=1)
    A = np.zeros((len(coords), K_TRUE))
    for k in range(K_TRUE):
        A[:,k] = (region == k) * np.exp(-d2[:,k]/5)
        A[:,k] *= rng.uniform(.8, 1.2, len(coords))
    return A / np.linalg.norm(A, axis=0)

def as_image(X):
    return nib.Nifti1Image(X.T.reshape(GRID + (len(X),)).astype('float32'), AFFINE)

def simulate(scenario, seed, make_images=True, strength=1., map_seed=None):
    rng = np.random.default_rng(seed)
    A = true_maps(seed if map_seed is None else map_seed)
    y = np.repeat([0,1], N_SUBJECTS//2)
    rng.shuffle(y)
    X = np.empty((N_SUBJECTS, 2, N_RUNS*RUN_LENGTH, np.prod(GRID)))
    clean = np.empty_like(X)
    for s in range(N_SUBJECTS):
        nuisance = np.exp(rng.normal(0, .18, K_TRUE))
        for lang in range(2):
            amplitude = nuisance.copy()
            rho = .15
            if lang == 1 and y[s] == 1:
                if scenario in ['amplitude', 'mixed']: amplitude[0] *= (1 + .65*strength)
                if scenario in ['coupling', 'mixed']: rho = .15 + .55*strength
            cov = np.eye(K_TRUE)
            cov[0,1] = cov[1,0] = rho
            runs=[]
            for run in range(N_RUNS):
                z = rng.normal(size=(RUN_LENGTH+40,K_TRUE)) @ np.linalg.cholesky(cov).T
                for t in range(1,len(z)):
                    z[t] = .35*z[t-1] + np.sqrt(1-.35**2)*z[t]
                runs.append(z[40:] * amplitude)
            clean[s,lang] = np.concatenate(runs) @ A.T
            X[s,lang] = clean[s,lang] + rng.normal(0,NOISE_SD,clean[s,lang].shape)
    imgs = [[as_image(X[s,l]) for l in range(2)] for s in range(N_SUBJECTS)] if make_images else None
    return dict(X=X, clean=clean, imgs=imgs, y=y, A=A)



def extract_features(Z):
    # Z: subject × condition × concatenated time × component.
    output={}
    for language in LANGUAGES:
        z = Z[:,0] if language == 'L1' else Z[:,1] if language == 'L2' else Z[:,1]-Z[:,0]
        rows=[]; edges=[]
        for subject_z in z:
            run_static=[]; run_edges=[]
            for r in range(N_RUNS):
                a = subject_z[r*RUN_LENGTH:(r+1)*RUN_LENGTH]
                sd = a.std(axis=0,ddof=1)
                lag = np.array([np.corrcoef(a[:-1,k],a[1:,k])[0,1] for k in range(K_FIT)])
                run_static.append(np.r_[a.mean(axis=0),sd,np.log(np.maximum(sd,1e-8)),lag])
                corr = np.corrcoef(a,rowvar=False)
                run_edges.append(np.arctanh(np.clip(corr[np.triu_indices(K_FIT,1)],-.999999,.999999)))
            rows.append(np.mean(run_static,axis=0))
            edges.append(np.mean(run_edges,axis=0))
        static=np.array(rows); connectivity=np.array(edges)
        output[language]={'static':static, 'connectivity':connectivity,
                          'combined':np.c_[static,connectivity]}
    # Run-wise summaries avoid lag-1 products across run boundaries and
    # correlations driven by between-run offsets. Confirm the empirical convention.
    assert all(np.isfinite(a).all() for d in output.values() for a in d.values())
    return output

def spatial_match(A, B):
    a=A-A.mean(axis=0); b=B-B.mean(axis=0)
    a/=np.linalg.norm(a,axis=0); b/=np.linalg.norm(b,axis=0)
    signed=a.T@b
    i,j=linear_sum_assignment(-np.abs(signed))
    return i,j,signed[i,j]

class ICAFitError(RuntimeError):
    pass

class AuditedCanICA(CanICA):
    def _unmix_components(self, components):
        if self.threshold is not None:
            raise ValueError('AuditedCanICA requires threshold=None in this notebook.')
        rng=check_random_state(self.random_state)
        seeds=rng.randint(np.iinfo(np.int32).max,size=self.n_init)
        self.restart_log_=[]; candidates=[]
        for algorithm in ['parallel', 'deflation']:
            # Prespecified optimizer fallback; never uses decoding outcomes.
            for seed in seeds:
                restart=len(self.restart_log_)
                with warnings.catch_warnings(record=True) as caught:
                    warnings.simplefilter('always')
                    _,_,sources,n_iter=fastica(np.asarray(components,dtype=np.float64),
                        whiten='arbitrary-variance',fun='cube',random_state=int(seed),
                        algorithm=algorithm,max_iter=ICA_MAX_ITER,tol=ICA_TOL,return_n_iter=True)
                converged=(n_iter < ICA_MAX_ITER) and not any(issubclass(w.category,ConvergenceWarning) for w in caught)
                maps=sources.T
                accepted=converged and np.isfinite(maps).all()
                self.restart_log_.append(dict(restart=restart,seed=int(seed),algorithm=algorithm,
                    n_iter=int(n_iter),hit_iteration_cap=bool(n_iter >= ICA_MAX_ITER),accepted=bool(accepted),warnings=[str(w.message) for w in caught]))
                if accepted:candidates.append((float(np.abs(maps).sum(axis=1).max()),restart,maps))
            if candidates:break
        if not candidates:
            raise ICAFitError('All spatial ICA restarts failed convergence/finite-value checks.')
        _,self.selected_restart_,maps=min(candidates,key=lambda item:item[0])
        self.components_=maps.astype(self.components_.dtype)
        for component in self.components_:
            if component.max() < -component.min(): component *= -1
        if hasattr(self,'masker_'):
            self.components_img_=self.masker_.inverse_transform(self.components_)

fit_log=[]
def fit_maps_checked(sim, ids, method, seed, context):
    start=time.perf_counter();ids=np.asarray(ids,dtype=int)
    model=None
    record=dict(**context,method=method,fit_ids=ids.tolist(),seed=int(seed))
    try:
        with warnings.catch_warnings(record=True) as caught:
            warnings.simplefilter('always')
            if method=='PCA':
                model=PCA(n_components=K_FIT,svd_solver='full')
                model.fit(sim['X'][ids].reshape(-1,np.prod(GRID)))
                A=model.components_.T;offset=model.mean_
            else:
                cls=AuditedCanICA if method=='AuditedCanICA' else CanICA
                model=cls(mask=MASK,n_components=K_FIT,do_cca=True,threshold=None,
                    n_init=CANICA_N_INIT,smoothing_fwhm=None,standardize=False,detrend=False,
                    random_state=seed,n_jobs=1)
                model.fit([sim['imgs'][s][l] for s in ids for l in range(2)])
                A=model.components_img_.get_fdata().reshape(-1,K_FIT)
                offset=np.zeros(A.shape[0])
            record['warnings']=[dict(category=w.category.__name__,message=str(w.message)) for w in caught]
            if any(issubclass(w.category,ConvergenceWarning) for w in caught):
                raise ICAFitError('Stock fit emitted convergence warning(s).')
        norms=np.linalg.norm(A,axis=0)
        if np.any(norms<1e-10) or np.linalg.matrix_rank(A)<K_FIT:raise ICAFitError('Degenerate maps.')
        record['status']='accepted'
        return dict(A=A/norms,offset=offset,fit_ids=ids.copy())
    except ICAFitError as error:
        record['status']='failed';record['error']=str(error)
        raise
    finally:
        record['seconds']=time.perf_counter()-start
        record['restarts']=getattr(model,'restart_log_',[])
        record['selected_restart']=getattr(model,'selected_restart_',None)
        fit_log.append(record)

def project(sim, fit):
    return (sim['X']-fit['offset'])@np.linalg.pinv(fit['A'].T)


## 4. Signal-strength sweep
To make “component summaries” precise, the new primary features are **log component variances** (5 features), **Fisher-z correlations** (10 features), or their full concatenation (15 features). This removes redundant summaries and lets the combined classifier retain both families. All features are standardized within each classifier training fold; regularization is fixed, and there is no feature selection or hyperparameter search.

AUC is computed within each test fold and averaged by its number of positive–negative pairs. Five independent cohort pairs are a small demonstration. Failure counts and matched-cohort counts accompany the curves. The comparison is conditional on ICA convergence. The independent-reference design tests transfer to new participants sharing the simulated spatial maps; it is not a replication of the original within-cohort CV design.


In [ ]:
def mechanism_features(Z):
    variances=[]; connections=[]
    for subject in Z[:,1]:
        vv=[];cc=[]
        for run in range(N_RUNS):
            z=subject[run*RUN_LENGTH:(run+1)*RUN_LENGTH]
            vv.append(np.log(np.maximum(z.var(axis=0,ddof=1),1e-10)))
            c=np.corrcoef(z,rowvar=False)
            cc.append(np.arctanh(np.clip(c[np.triu_indices(K_FIT,1)],-.999999,.999999)))
        variances.append(np.mean(vv,axis=0));connections.append(np.mean(cc,axis=0))
    v,c=np.array(variances),np.array(connections)
    return {'Variance':v,'Connectivity':c,'Combined':np.c_[v,c]}

def decode(F,y,splits):
    scores=[];weights=[]
    for train,test in splits:
        model=make_pipeline(StandardScaler(),LogisticRegression(C=1.,solver='liblinear',
                                        class_weight='balanced',random_state=SEED))
        model.fit(F[train],y[train]);p=model.predict_proba(F[test])[:,1]
        scores.append(roc_auc_score(y[test],p));weights.append(sum(y[test]==0)*sum(y[test]==1))
    return float(np.average(scores,weights=weights))

rows=[];failed=[];examples={};fit_log=[]
t0=time.perf_counter()
with threadpool_limits(limits=1):
    for rep in range(N_SWEEP_REPS):
        for si,scenario in enumerate(['amplitude','coupling','mixed']):
            # Independent random streams, fixed across strengths within this cohort pair.
            seed=SEED+10000*rep+100*si
            map_seed=SEED+10000*rep
            for strength in STRENGTHS:
                ref=simulate(scenario,seed+1000000,strength=strength,map_seed=map_seed)
                ev=simulate(scenario,seed+2000000,strength=strength,map_seed=map_seed,make_images=False)
                assert np.allclose(ref['A'],ev['A'])
                splits=list(StratifiedKFold(N_SPLITS,shuffle=True,random_state=seed).split(ev['y'],ev['y']))
                # Known maps provide a reference without spatial decomposition error.
                oracle=dict(A=ev['A'],offset=np.zeros(np.prod(GRID)))
                for method in ['Known maps']+METHODS:
                    try:
                        fit=oracle if method=='Known maps' else fit_maps_checked(ref,np.arange(N_SUBJECTS),method,
                            seed,dict(rep=rep,scenario=scenario,strength=strength,scope='independent_reference',fold=-1))
                    except ICAFitError as err:
                        failed.append(dict(rep=rep,scenario=scenario,strength=strength,method=method,error=str(err)))
                        continue
                    Z=project(ev,fit)
                    for family,F in mechanism_features(Z).items():
                        rows.append(dict(rep=rep,scenario=scenario,strength=strength,method=method,
                                         family=family,auc=decode(F,ev['y'],splits)))
                    # Fixed example specified before inspecting performance, never replace on failure.
                    if rep==0 and scenario=='coupling' and strength==1.:
                        examples[method]=dict(sim=ev,fit=fit,Z=Z)
            print(f'Cohort pair {rep+1}/{N_SWEEP_REPS}, {scenario}: {time.perf_counter()-t0:.1f}s',flush=True)
sweep=pd.DataFrame(rows);failures=pd.DataFrame(failed)
print('Failed map fits:',len(failures))
if len(failures):display(failures)
# Compare methods on identical successful cohort pairs at each strength.
key=['rep','scenario','strength']
counts=sweep.groupby(key).method.nunique()
valid=counts[counts==3].reset_index()[key]
paired=sweep.merge(valid,on=key)
coverage=paired.groupby(['scenario','strength']).rep.nunique().unstack().reindex(
    index=['amplitude','coupling','mixed'],columns=STRENGTHS).fillna(0).astype(int)
print('Common successful cohort pairs (out of',N_SWEEP_REPS,'):');display(coverage)


## 5. Where the coupling signal appears
These plots use the prespecified first coupling cohort pair at strength 1. Maps and time courses are matched to generating maps by maximum absolute spatial correlation **for display only**. Matching is not used by any classifier. For PCA, a matched map can still be a mixture; matching does not establish source recovery. Trace amplitudes are standardized only for the time-course display. Covariance and variance plots use the original projected amplitudes.


In [ ]:
aligned={}
for method,e in examples.items():
    i,j,c=spatial_match(e['sim']['A'],e['fit']['A'])
    order=j[np.argsort(i)];sign=np.sign(c[np.argsort(i)]);sign[sign==0]=1
    aligned[method]=dict(A=e['fit']['A'][:,order]*sign,Z=e['Z'][...,order]*sign,
                         sim=e['sim'],match=np.abs(c).mean())
if len(aligned)<3:print('The prespecified ICA example failed; available methods are shown without replacing its seed.')
fig,axes=plt.subplots(len(aligned),K_FIT,figsize=(12,2.4*len(aligned)),squeeze=False)
for r,(method,e) in enumerate(aligned.items()):
    for k in range(K_FIT):
        m=e['A'][:,k].reshape(GRID)[:,:,1].T
        lim=max(np.abs(m).max(),1e-8)
        axes[r,k].imshow(m,cmap='RdBu_r',vmin=-lim,vmax=lim,origin='lower')
        axes[r,k].set_xticks([]);axes[r,k].set_yticks([])
        if r==0:axes[r,k].set_title(f'Matched source {k+1}')
        if k==0:axes[r,k].set_ylabel(method)
fig.suptitle('Generating-map projection and recovered spatial maps: toy central slices')
plt.tight_layout();plt.show()
# Noise-free generating time courses, recovered by projecting clean data onto true maps.
sim=next(iter(examples.values()))['sim']
truth=sim['clean']@np.linalg.pinv(sim['A'].T)
subject=int(np.flatnonzero(sim['y']==1)[0])
fig,axes=plt.subplots(K_FIT,1,figsize=(11,9),sharex=True)
for k,ax in enumerate(axes):
    z=truth[subject,1,:RUN_LENGTH,k];ax.plot((z-z.mean())/z.std(),color='black',lw=2,label='True source')
    for method,e in aligned.items():
        if method=='Known maps':continue
        z=e['Z'][subject,1,:RUN_LENGTH,k]
        ax.plot((z-z.mean())/z.std(),alpha=.8,label=method)
    ax.set_ylabel(f'Source {k+1}')
axes[0].legend(ncol=3,frameon=False);axes[-1].set_xlabel('Timepoint in run 1')
fig.suptitle('Matched time courses for one prespecified group-1 participant (display z-scores)')
plt.tight_layout();plt.show()

fig,axes=plt.subplots(len(aligned),3,figsize=(12,3.2*len(aligned)),squeeze=False)
for r,(method,e) in enumerate(aligned.items()):
    Z=e['Z'][:,1];y=e['sim']['y'];cov=[];corr=[]
    for z in Z:
        chunks=[z[t*RUN_LENGTH:(t+1)*RUN_LENGTH] for t in range(N_RUNS)]
        cov.append(np.mean([np.cov(a,rowvar=False) for a in chunks],axis=0))
        corr.append(np.mean([np.corrcoef(a,rowvar=False) for a in chunks],axis=0))
    cov=np.array(cov);corr=np.array(corr)
    g0=cov[y==0].mean(0);g1=cov[y==1].mean(0)
    x=np.arange(K_FIT)
    axes[r,0].bar(x-.18,np.diag(g0),width=.36,label='Group 0',color='#7399BC')
    axes[r,0].bar(x+.18,np.diag(g1),width=.36,label='Group 1',color='#D57939')
    axes[r,0].set_ylabel(method+'\nMean variance');axes[r,0].set_xticks(x,x+1)
    for col,M in [(1,g1-g0),(2,corr[y==1].mean(0)-corr[y==0].mean(0))]:
        lim=max(np.abs(M).max(),1e-8)
        im=axes[r,col].imshow(M,cmap='RdBu_r',vmin=-lim,vmax=lim)
        axes[r,col].set_xticks(x,x+1);axes[r,col].set_yticks(x,x+1)
        fig.colorbar(im,ax=axes[r,col],shrink=.7)
    if r==0:
        axes[r,0].set_title('Component variances');axes[r,0].legend(frameon=False)
        axes[r,1].set_title('Group 1 − Group 0 covariance')
        axes[r,2].set_title('Group 1 − Group 0 correlation')
fig.suptitle('The same coupling manipulation in three coordinate systems; descriptive example')
plt.tight_layout();plt.show()


## 6. Decoding as signal strength increases
Lines show means across common successful cohort pairs. Shading is ±1 SD across those pairs, **not a confidence interval**; counts are printed above. A single successful pair has no band. Read these as illustrative curves rather than a ranking of methods. The zero-strength points are the learned-map null check. Failed and successful fits are all retained in the logs.


The set of successful cohort pairs can change across strengths. Consequently, changes along a curve can reflect both signal strength and which ICA fits converged. Use the coverage table when interpreting the curves.

In [ ]:
colors={'PCA':'#3274A1','AuditedCanICA':'#D57939','Known maps':'#777777'}
summary=paired.groupby(['scenario','family','method','strength']).auc.agg(['mean','std','count']).reset_index()
fig,axes=plt.subplots(3,3,figsize=(12,10),sharex=True,sharey=True)
for r,scenario in enumerate(['amplitude','coupling','mixed']):
    for col,family in enumerate(['Variance','Connectivity','Combined']):
        ax=axes[r,col]
        for method in ['Known maps']+METHODS:
            q=summary[(summary.scenario==scenario)&(summary.family==family)&(summary.method==method)].sort_values('strength')
            x=q.strength.to_numpy();m=q['mean'].to_numpy();sd=q['std'].to_numpy()
            ax.plot(x,m,marker='o',label=method,color=colors.get(method),ls='--' if method=='Known maps' else '-')
            ax.fill_between(x,np.maximum(0,m-sd),np.minimum(1,m+sd),alpha=.12,color=colors.get(method))
        ax.axhline(.5,color='gray',ls=':',lw=1);ax.set_ylim(0,1.03)
        if r==0:ax.set_title(family)
        if col==0:ax.set_ylabel(scenario.capitalize()+'\nHeld-out AUC')
        if r==2:ax.set_xlabel('Manipulation strength')
axes[0,0].legend(frameon=False,fontsize=8)
fig.suptitle('Signal-strength sweep: independent reference maps, held-out classification')
plt.tight_layout();plt.show()
print('New coupling endpoint (different features and evaluation design from the preserved result):')
display(summary[(summary.scenario=='coupling')&(summary.strength==1)][['family','method','mean','std','count']])


## Interpretation and limits
- The exact two-source calculation establishes that covariance information can move between diagonal and off-diagonal entries under rotation. It does not imply that spatial ICA always recovers the true sources.
- The historical coupling table is preserved, while the new sweep tests whether that pattern persists as signal weakens. New results are allowed to differ; do not tune seeds or acceptance rules to recover the previous table.
- The simulation has shared toy maps and relatively strong endpoint effects. It is not anatomical validation, evidence about proficiency in Onila's participants, or an audit of the PCA paper.
- The known-map curves use noisy observations projected onto generating maps, not noiseless latent sources. They are a reference, not a formal upper bound on finite-sample decoding accuracy.
- Combined features retain both families, but fixed regularization and limited samples can still reduce performance. The two feature families have different dimensions.
- ICA stopping checks do not guarantee identifiable or biologically valid components. Common-cohort comparisons remain conditional on convergence; increase replicate count and examine failures before making quantitative method claims.
- Independent reference maps intentionally make this a representation experiment. The earlier all-participant versus fold-fitted comparison remains a separate sensitivity question.

Optional export below saves the data behind these figures and all fitting diagnostics. No external data downloads are required to run the notebook.


In [ ]:
SAVE_RESULTS=False
if SAVE_RESULTS:
    from pathlib import Path
    out=Path('pca_ica_mechanism_results');out.mkdir(exist_ok=True)
    for name,table in [('original_coupling',original_coupling),('sweep_all',sweep),
                       ('sweep_paired',paired),('summary',summary),('failures',failures),('coverage',coverage)]:
        table.to_csv(out/(name+'.csv'),index=(name=='coverage'))
    (out/'fit_log.json').write_text(json.dumps(fit_log,indent=2))
    (out/'settings.json').write_text(json.dumps(dict(seed=SEED,n_subjects=N_SUBJECTS,
        strengths=STRENGTHS,reps=N_SWEEP_REPS,noise_sd=NOISE_SD,n_splits=N_SPLITS,
        ica_max_iter=ICA_MAX_ITER,ica_tol=ICA_TOL,n_init=CANICA_N_INIT,
        numpy=np.__version__,sklearn=sklearn.__version__,nilearn=nilearn.__version__),indent=2))
    print('Saved',out.resolve())
